# train_ensemble_full — multi-view LambdaMART on R → full-corpus TREC22

Deliberately DIVERSE features, each self-consistent on its own representation (no train/inference mismatch,
but different models read different *views*): retrieval (BM25/dense/RRF) + **clf_R** (eligibility view) +
**clf_topic** (topicality view) + **eligibility judge** + **topicality judge**. `reranker_v3` is dropped
(it was a clf_R clone, r=0.69). clf_topic / topicality auto-included if present. Tuning on TREC21 CV only.


## Setup (Colab — GPU for cross-encoder scoring; cached after first run)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q lightgbm pytrec_eval datasets transformers pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd, torch, lightgbm as lgb
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from ctmatch.experiments import (ExperimentConfig, load_corpus, load_eval, cross_encoder_scores,
                                 relevant_index, resolve_ckpt, pytrec_metrics, ndcg_at_k, log_result)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
POOL_TAG = 'nqs'   # 'R' for the original hybrid pool
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag=POOL_TAG)   # eligibility view (elig_first)

In [ ]:
# Retrieval feats + both judges (topicality optional).
corpus_ids, corpus_fields = load_corpus(cfg); id2fields = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(cfg, ['trec21','kz','trec22'])
pool = json.load(open(cfg.pool_path()))
rfeat = {}
for l in open(cfg.feat_file('retrieval_feats')):
    r = json.loads(l); rfeat[(r['source'], r['topic_id'], r['doc_id'])] = r
llm = {}
for l in open(cfg.feat_file('llm_scores')):
    r = json.loads(l); llm[(r['source'], r['topic_id'], r['doc_id'])] = r['llm_score']
TOPI_PATH = cfg.feat_file('topicality'); USE_TOPI = os.path.exists(TOPI_PATH); topi = {}
if USE_TOPI:
    for l in open(TOPI_PATH):
        r = json.loads(l); topi[(r['source'], r['topic_id'], r['doc_id'])] = r['topicality']
print('topicality judge:', USE_TOPI)
# Prefer LLM-expanded condition_match (written by rerank_condition_match.ipynb); fall back to original.
CM_PATH = cfg.feat_file('condition_match_exp')
if not os.path.exists(CM_PATH):
    CM_PATH = cfg.feat_file('condition_match')
USE_CM = os.path.exists(CM_PATH); cm = {}
if USE_CM:
    for l in open(CM_PATH):
        r = json.loads(l); cm[(r['source'], r['topic_id'], r['doc_id'])] = r['condition_match']
print('condition_match:', os.path.basename(CM_PATH) if USE_CM else 'not found')

In [ ]:
# Cross-encoder VIEWS: clf_R on elig_first, clf_topic on topic_first. Each scored with ITS repr. Cached per model.
def ce_feats(name, ckpt, mcfg):
    path = cfg.ce_cache_path(name)
    if os.path.exists(path): return np.load(path, allow_pickle=True)['d'].item()
    rk = resolve_ckpt(cfg, ckpt); tok = AutoTokenizer.from_pretrained(rk)
    m = AutoModelForSequenceClassification.from_pretrained(rk).to(device).eval()
    ridx = relevant_index(m); pidx = next((i for i,v in m.config.id2label.items() if 'partial' in v.lower()), None)
    out = {}
    for s in pool:
        for t, docs in tqdm(pool[s].items(), desc=f'{name} {s}', leave=False):
            ids = [d for d in docs if d in id2fields]; fields = [id2fields[d] for d in ids]
            rel = cross_encoder_scores(m, tok, sets[s]['topic2text'][t], fields, mcfg, ridx)
            par = cross_encoder_scores(m, tok, sets[s]['topic2text'][t], fields, mcfg, pidx) if pidx is not None else [0.]*len(ids)
            for d, rr, pp in zip(ids, rel, par): out[(s,t,d)] = (rr, pp)
    del m; torch.cuda.empty_cache(); np.savez(path, d=out); return out
clf_f = ce_feats('clf_R', cfg.clf_ckpt, cfg)
try:
    # load from the local Drive dir (always present after training) — no HF push needed
    clft_f = ce_feats('clf_topic', 'models/clf_topic', cfg.with_(repr_strategy='topic_first'))
except Exception as e:
    print('clf_topic not available yet -> skipping its features:', type(e).__name__); clft_f = None


In [ ]:
# Feature set (order-safe via dict). v2_rel dropped. clf_topic / topicality included when present.
FEATURES = ['bm25','bm25_rank','dense','dense_rank','rrf','clf_rel','clf_partial']
if clft_f is not None: FEATURES += ['clf_topic_rel','clf_topic_partial']
FEATURES += ['llm_yesno']
if USE_TOPI: FEATURES += ['topicality']
if USE_CM: FEATURES += ['condition_match']
print('FEATURES:', FEATURES)
def featvec(s,t,d):
    rf = rfeat.get((s,t,d), {}); cr,cp = clf_f.get((s,t,d),(0.,0.))
    v = {'bm25':rf.get('bm25',0.),'bm25_rank':rf.get('bm25_rank',cfg.cand_k),'dense':rf.get('dense',0.),
         'dense_rank':rf.get('dense_rank',cfg.cand_k),'rrf':rf.get('rrf',0.),'clf_rel':cr,'clf_partial':cp,
         'llm_yesno':llm.get((s,t,d), cfg.llm_floor)}
    if clft_f is not None: tr,tp = clft_f.get((s,t,d),(0.,0.)); v['clf_topic_rel']=tr; v['clf_topic_partial']=tp
    if USE_TOPI: v['topicality']=topi.get((s,t,d),0.)
    if USE_CM: v['condition_match']=cm.get((s,t,d),0.)
    return [v[f] for f in FEATURES]
def build(s):
    X,y,g = [],[],[]; rel = sets[s]['rel_dict']
    for t, docs in pool[s].items():
        docs = [d for d in docs if d in id2fields]; g.append(len(docs))
        for d in docs: X.append(featvec(s,t,d)); y.append(int(rel[t].get(d,0)))
    return np.array(X,dtype=np.float32), np.array(y), g
X21,y21,g21 = build('trec21'); Xkz,ykz,gkz = build('kz'); Xte,yte,gte = build('trec22')
Xtr = np.vstack([X21,Xkz]); ytr = np.concatenate([y21,ykz]); gtr = g21+gkz
print('train', Xtr.shape, '| tune-on trec21', X21.shape, '| test', Xte.shape)


In [ ]:
# Tune on TREC21 ONLY: backward feature selection + num_leaves.
topic_of = np.concatenate([[i]*c for i,c in enumerate(g21)]); nt=len(g21)
fold = np.random.default_rng(cfg.seed).integers(0,5,nt)
def _ndcg(y,s):
    o=np.argsort(-s)[:10]; gg=(2.0**y[o]-1); d=1/np.log2(np.arange(2,2+len(o)))
    idcg=((2.0**np.sort(y)[::-1][:10]-1)/np.log2(np.arange(2,2+min(10,len(y))))).sum()
    return (gg*d).sum()/idcg if idcg>0 else 0.0
def cv(cols, nl):
    vals=[]
    for f in range(5):
        trm=np.isin(topic_of,[i for i in range(nt) if fold[i]!=f]); vm=~trm
        gt=[g21[i] for i in range(nt) if fold[i]!=f]; gv=[g21[i] for i in range(nt) if fold[i]==f]
        b=lgb.train({'objective':'lambdarank','metric':'ndcg','ndcg_eval_at':[10],'num_leaves':nl,
                     'min_data_in_leaf':20,'learning_rate':0.05,'lambda_l2':1.0,'verbose':-1},
                    lgb.Dataset(X21[trm][:,cols],y21[trm],group=gt),num_boost_round=50)
        p=b.predict(X21[vm][:,cols]); i0=0
        for gg in gv: vals.append(_ndcg(y21[vm][i0:i0+gg],p[i0:i0+gg])); i0+=gg
    return float(np.mean(vals))
cols=list(range(len(FEATURES))); cur=cv(cols,15)
while len(cols)>1:
    cand=[(cv([c for c in cols if c!=j],15),j) for j in cols]; bv,bj=max(cand)
    if bv>cur+1e-4: cur=bv; cols.remove(bj); print('drop',FEATURES[bj],'-> CV',round(bv,4))
    else: break
NL=max([7,15,31], key=lambda nl: cv(cols,nl)); SEL=cols
print('SELECTED', [FEATURES[c] for c in SEL], '| num_leaves', NL, '| trec21 CV', round(cv(SEL,NL),4))


In [ ]:
# Final: train on TREC21+KZ with selected features; TREC22 touched ONCE.
booster = lgb.train({'objective':'lambdarank','metric':'ndcg','ndcg_eval_at':[10],'num_leaves':NL,
                     'min_data_in_leaf':20,'learning_rate':0.05,'lambda_l2':1.0,'verbose':-1},
                    lgb.Dataset(Xtr[:,SEL],ytr,group=gtr), num_boost_round=50)
pred = booster.predict(Xte[:,SEL]); run={}; i0=0
for t in pool['trec22']:
    for d in [d for d in pool['trec22'][t] if d in id2fields]: run.setdefault(t,{})[d]=float(pred[i0]); i0+=1
qrels = {t:{d:int(r) for d,r in sets['trec22']['rel_dict'][t].items()} for t in run}
metrics = pytrec_metrics(run, qrels, k=10)
log_result(cfg, experiment='train_ensemble_full', split='trec22', metrics=metrics,
           extra={'num_leaves':NL,'features':[FEATURES[c] for c in SEL]})
print('TREC22 full-corpus:', metrics)
print('importances:', dict(zip([FEATURES[c] for c in SEL], booster.feature_importance().tolist())))


# Persist the fitted booster + the ORDERED selected-feature list so the external TREC23 test
# applies THIS exact ensemble (no re-fit, no re-selection -> no drift). Also makes the TREC22
# headline reproducible.
os.makedirs(cfg.path('models'), exist_ok=True)
booster.save_model(cfg.path(f'models/ensemble_{cfg.pool_tag}.txt'))
json.dump([FEATURES[c] for c in SEL], open(cfg.path(f'models/ensemble_{cfg.pool_tag}_features.json'), 'w'))
print('persisted booster + feature order ->', cfg.path(f'models/ensemble_{cfg.pool_tag}.txt'))


In [ ]:
import pytrec_eval
per = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut.10'}).evaluate(run)
vals = np.array([per[t]['ndcg_cut_10'] for t in per])
boot = [np.mean(np.random.choice(vals, len(vals), replace=True)) for _ in range(10000)]
print(f'TREC22 NDCG@10 = {vals.mean():.4f}  95% CI [{np.percentile(boot,2.5):.4f}, {np.percentile(boot,97.5):.4f}]  vs h2oloo 0.6125')


In [ ]:
# Per-topic NDCG@10 (TREC22) — the §8a rescued topics, and a diff vs the other pool if present.
import pytrec_eval
pt = {t: v['ndcg_cut_10'] for t,v in pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut.10'}).evaluate(run).items()}
os.makedirs(cfg.path('results'), exist_ok=True)
with open(cfg.path(f'results/per_topic_ndcg_{cfg.pool_tag}.jsonl'),'w') as f:
    for t,v in pt.items(): f.write(json.dumps({'topic':t,'ndcg@10':round(v,4)})+'\n')
S8A = ['41','43','40','11','8','28','32','10']   # implicit-diagnosis topics NQS targets
print('§8a topics NDCG@10:', {t: round(pt[t],3) for t in S8A if t in pt})
other = 'R' if cfg.pool_tag=='nqs' else 'nqs'; op = cfg.path(f'results/per_topic_ndcg_{other}.jsonl')
if os.path.exists(op):
    ot = {json.loads(l)['topic']: json.loads(l)['ndcg@10'] for l in open(op)}
    diff = sorted(((t, round(pt[t]-ot.get(t,0),3)) for t in pt if t in ot), key=lambda x:-x[1])
    print(f'biggest gainers vs {other} pool:', diff[:8])
    print(f'biggest losers  vs {other} pool:', diff[-5:])
else:
    print(f'(run the {other} pool too to get the per-topic diff)')


## Reading it
- Multi-view: watch whether **clf_topic** and **topicality** survive selection and add over the eligibility views.
- If the two orthogonal (topicality) features carry weight, the diversity is back — *principled* this time.
- Report TREC22 (touched once) + TREC21 CV (honest secondary); update deep dive §7/§2h.
